# Rod-bundle PINN — step-by-step walkthrough

Trains the physics-informed neural network on your OpenFOAM data and checks it
against the 2023 DNS paper. Run the cells **top to bottom** (Shift+Enter).

It uses the same code as `src/train.py` and `src/evaluate.py`, so a result here is
identical to a terminal run. Kernel: pick **enygf**.

Before starting you need `digitized_data/cfd_generated/cfd_profiles.csv`
(from `cfd/openfoam/extract_profiles.py`).

In [ ]:
import os, sys
from pathlib import Path

# Work from ml/ so the paths in configs/default.yaml resolve
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd() / "src"))

import yaml, torch, pandas as pd, matplotlib.pyplot as plt
import train as trainer
import evaluate as evaluator

cfg = yaml.safe_load(open("configs/default.yaml"))
print("working dir:", Path.cwd())
print("python:", sys.executable)
print("device:", "GPU (CUDA)" if torch.cuda.is_available() else "CPU")

## Step 1 — Look at the training data

The CFD profiles the network learns from. `xi` is the path the DNS paper plots along:
rod surface in the narrow gap → gap centre → subchannel centre → back to the rod at 45°.

In [ ]:
df = pd.read_csv(cfg["paths"]["cfd_profiles"])
print(df.groupby("quantity").size().rename("points"))

path = df[df["line"].isin(["seg1", "seg2", "seg3"])]
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, q, label in zip(axes, ["w", "nut", "k"], ["axial velocity w [m/s]", "eddy viscosity nu_t [m2/s]", "TKE k [m2/s2]"]):
    g = path[path["quantity"] == q].sort_values("xi")
    ax.plot(g["xi"] / cfg["geometry"]["dh_dns"], g["value"], ".", ms=2)
    ax.set_xlabel("xi / Dh"); ax.set_title(label)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, bc in zip(axes, ["isoT", "isoFlux"]):
    for case, g in path[(path["quantity"] == "T") & (path["bc"] == bc)].groupby("case_id"):
        g = g.sort_values("xi")
        ax.plot(g["xi"] / cfg["geometry"]["dh_dns"], g["value"], ".", ms=2, label=case)
    ax.set_xlabel("xi / Dh"); ax.set_title(f"temperature, {bc}"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

## Step 2 — Choose the run length

`QUICK = True` runs 100 epochs (~30 s) to check everything works.
`QUICK = False` runs the full schedule (~6000 epochs, ~30 min on a laptop CPU).

In [ ]:
QUICK = False

if QUICK:
    cfg["training"].update(epochs_momentum=50, epochs_joint=50, print_every=10)
n = cfg["training"]["epochs_momentum"] + cfg["training"]["epochs_joint"]
print(f"{n} epochs  (~{n * 0.3 / 60:.0f} min at 0.3 s/epoch)")

## Step 3 — Train

- **Phase 1** (flow only): learns axial velocity, eddy viscosity and TKE from the CFD,
  while satisfying the momentum equation and holding the bulk velocity at 1 m/s.
- **Phase 2** (flow + temperature): adds the 8 temperature fields and the energy equation.

Every term should trend down, with noise. `dpdz` is the learned pressure gradient; the
CFD value is **0.188**.

In [ ]:
model = trainer.train(cfg)

## Step 4 — Loss curves

What to look for:
- **Every curve falls** over the run. A term that stays flat is not being learned.
- **`pde_*` terms** (equation residuals) fall a lot more slowly than `data_*` terms. That's normal.
- **A jump at the phase boundary** is expected: temperature switches on there.
- **`dpdz`** should settle near the CFD's 0.188. The DNS equivalent is about 0.207.

In [ ]:
hist = pd.read_csv(cfg["paths"]["loss_history"])
terms = [c for c in hist.columns if c not in ("epoch", "phase", "total", "dpdz")]
switch = cfg["training"]["epochs_momentum"]

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for t in terms:
    s = hist[["epoch", t]].dropna()
    axes[0].semilogy(s["epoch"], s[t].rolling(20, min_periods=1).mean(), label=t)
axes[0].axvline(switch, color="gray", ls=":")
axes[0].set_xlabel("epoch"); axes[0].set_title("loss terms (20-epoch moving average)")
axes[0].legend(fontsize=7, ncol=2)

axes[1].plot(hist["epoch"], hist["dpdz"], label="PINN")
axes[1].axhline(0.188, color="k", ls="--", label="OpenFOAM CFD (0.188)")
axes[1].axhline(4 * 0.0637**2 / cfg["geometry"]["dh_cell"], color="r", ls=":", label="from DNS u_tau (0.207)")
axes[1].set_xlabel("epoch"); axes[1].set_title("learned pressure gradient dp/dz [m/s2]"); axes[1].legend()
plt.tight_layout(); plt.show()

## Step 5 — Evaluate

1. **Fit to the CFD** it trained on (`rmse_rel` = error / largest value; below ~0.05 is a good fit).
2. **Nusselt numbers vs. the DNS** (Table 1 of the 2023 paper). This is the independent check:
   the network never saw these numbers.
3. **Digitized DNS figures**, if you have filled in `digitized_data/dns_*.csv`.

In [ ]:
fit, nusselt, dns = evaluator.evaluate(cfg)

In [ ]:
from IPython.display import Image, display
for png in sorted(Path(cfg["paths"]["plots_dir"]).glob("*.png")):
    print(png.name)
    display(Image(filename=str(png)))

## If it doesn't converge well

Change one thing at a time in `configs/default.yaml` (or in `cfg` above) and rerun from Step 2:

- **Losses still falling at the end** → train longer: increase `epochs_joint`.
- **Curves noisy or jumping** → lower `lr` (e.g. `5.0e-4`).
- **One data term stuck while its PDE term falls** → raise that data weight in `loss_weights`.
- **Temperature fit poor for Pr = 7** → expected to be hardest: thinnest thermal layer. Try more epochs first.

Write down each change and its effect. That log is part of your methodology section.